In [33]:
import pandas as pd
import glob, re
from datetime import datetime

### see what we have already

In [6]:
ed_old = pd.read_csv('raw_data/event_details_2009-21.csv')

In [7]:
ed_old['date'] = pd.to_datetime(ed_old.date)

In [8]:
ed_old.dtypes

region               object
club                 object
type                 object
distance            float64
date         datetime64[ns]
finishers             int64
dnf                 float64
eid                   int64
dtype: object

In [9]:
[min(ed_old.date), max(ed_old.date)]

[Timestamp('2009-01-01 00:00:00'), Timestamp('2021-12-31 00:00:00')]

In [10]:
ed_old. \
    assign(yr=ed_old.date.dt.year). \
    groupby('yr').size()

yr
2009    411
2010    532
2011    622
2012    626
2013    634
2014    637
2015    719
2016    727
2017    799
2018    928
2019    924
2020    506
2021    705
dtype: int64

### smush in new

In [61]:
def csv_smush(fl_list):
    dat = pd.concat([pd.read_csv(f) for f in fl_list], ignore_index=True)
    return dat

For event_details, normalize date formats between files

In [14]:
event_details_lst = [pd.read_csv(f) for f in glob.glob('raw_data/event_details*')]
for d in event_details_lst:
    d['date'] = pd.to_datetime(d.date)
    #event_details['date'] = pd.to_datetime(event_details.date)

In [18]:
event_details = pd.concat(event_details_lst)

For event results, only the per-year files need hours/mins added

In [39]:
glob.glob('raw_data/event_results*.csv')

['raw_data/event_results_2009-21.csv',
 'raw_data/event_results_2024.csv',
 'raw_data/event_results_2023.csv',
 'raw_data/event_results_2022.csv']

In [60]:
new_ed_fls = [s for s in glob.glob('raw_data/event_results*.csv') if re.match(r'.*[0-9]{4}\.csv$', s)]

In [63]:
event_results_new = csv_smush(new_ed_fls)
# extract hours and minutes from hh:mm string and store as numeric
event_results_new['hours'] = pd.to_numeric(event_results_new.time.str.replace(':.*$', '', regex=True))
event_results_new['minutes'] = pd.to_numeric(event_results_new.time.str.replace('^.*:', '', regex=True))

In [64]:
event_results_new

,cert,rusa,name,club / acp code,time,medal,eid,hours,minutes
0,923390,5844,"AURIEMMA, Philip J",San Francisco Randonneurs / 905030,09:47,NaN,15716,9,47
1,923391,10133,"BERNHARDT, Luis",San Francisco Randonneurs / 905030,09:43,NaN,15716,9,43
2,923392,16644,"BURGHART, Dan",Randonneurs USA / 905095,09:51,Y,15716,9,51
3,923393,16048,"CARILLO, Christian R",Randonneurs USA / 905095,08:58,NaN,15716,8,58
4,923394,14197,"MANN, Deirdre",Adobo Velo Filipino - American Cycling Club / ...,09:47,NaN,15716,9,47
...,...,...,...,...,...,...,...,...,...
21339,295836,12169,"WIECHERS, Brenda J",Ontario Randonneurs - Huron / 011804,18:06,NaN,14668,18,6
21340,RUSA-B17226,4500,"AKBARIAN, Hamid",Northern Virginia Randonneurs / 946020,09:00,NaN,14682,9,0
21341,RUSA-B17227,3255,"TOSOLINI, Andrea",Gainesville Cycling Club / 909005,09:00,NaN,14682,9,0
21342,RUSA-P20031,8010,"RUSSELL, Amy L",Heart of Texas Randonneurs / 943049,04:47,NaN,14698,4,47


In [65]:
event_results_old = pd.read_csv('raw_data/event_results_2009-21.csv')

In [66]:
event_results_old

,cert,rusa,name,club / acp code,time,medal,eid,hours,minutes
0,373131,8037,"AGATEP, Scott",Davis Bike Club / 905014,09:44,NaN,5426,9,44
1,373132,6328,"ALBRIGHT, Dean",Davis Bike Club / 905014,09:20,Y,5426,9,20
2,373133,5776,"AMEEN, Sol",Davis Bike Club / 905014,11:20,Y,5426,11,20
3,373134,6151,"ANDERSEN, Carl",San Francisco Randonneurs / 905030,08:47,NaN,5426,8,47
4,373135,8313,"BACKMAN, Steve C",Santa Rosa Cycling Club / 905048,08:27,NaN,5426,8,27
...,...,...,...,...,...,...,...,...,...
87114,RUSA-B05543,4495,"TYER, Vickie",Lone Star Randonneurs / 943026,13:44,NaN,3384,13,44
87115,RUSA-B05544,64,"THOMAS, Mark",Seattle International Randonneurs / 947018,13:44,NaN,3384,13,44
87116,RUSA-B05545,1576,"PHELPS, Robin",Rocky Mountain Cycling Club / 906002,13:44,NaN,3384,13,44
87117,RUSA-B05546,2299,"PHELPS, Val",Rocky Mountain Cycling Club / 906002,13:44,NaN,3384,13,44


In [67]:
event_results = pd.concat([event_results_old, event_results_new])
event_results

,cert,rusa,name,club / acp code,time,medal,eid,hours,minutes
0,373131,8037,"AGATEP, Scott",Davis Bike Club / 905014,09:44,NaN,5426,9,44
1,373132,6328,"ALBRIGHT, Dean",Davis Bike Club / 905014,09:20,Y,5426,9,20
2,373133,5776,"AMEEN, Sol",Davis Bike Club / 905014,11:20,Y,5426,11,20
3,373134,6151,"ANDERSEN, Carl",San Francisco Randonneurs / 905030,08:47,NaN,5426,8,47
4,373135,8313,"BACKMAN, Steve C",Santa Rosa Cycling Club / 905048,08:27,NaN,5426,8,27
...,...,...,...,...,...,...,...,...,...
21339,295836,12169,"WIECHERS, Brenda J",Ontario Randonneurs - Huron / 011804,18:06,NaN,14668,18,6
21340,RUSA-B17226,4500,"AKBARIAN, Hamid",Northern Virginia Randonneurs / 946020,09:00,NaN,14682,9,0
21341,RUSA-B17227,3255,"TOSOLINI, Andrea",Gainesville Cycling Club / 909005,09:00,NaN,14682,9,0
21342,RUSA-P20031,8010,"RUSSELL, Amy L",Heart of Texas Randonneurs / 943049,04:47,NaN,14698,4,47


we can directly smush events

In [68]:
events = csv_smush(glob.glob('raw_data/events*'))

In [69]:
events.to_csv('data/events.csv', index=False)
event_details.to_csv('data/event_details.csv', index=False)
event_results.to_csv('data/event_results.csv', index=False)